# 📖 Notebook 4: Advanced Gateway Patterns

You've seen the core gateway responsibilities (routing, rate limiting, auth, transformation). This notebook covers the **advanced patterns** that turn a basic gateway into a production-grade one:

| Pattern | What it solves |
|---------|----------------|
| **Circuit-breaker-style failover** | Stops sending traffic to a backend that keeps failing |
| **Request aggregation (BFF)** | Lets clients make one call instead of many |
| **Canary / weighted routing** | Ships a new version to a small % of users |
| **CORS** | Lets browsers from other origins call your API safely |
| **Observability (logging/tracing)** | Makes it possible to debug what went wrong, in which service |
| **SSL/TLS termination** | Terminates HTTPS at the edge so backends stay simple |

We'll follow the same 🚫 BAD → ✅ BETTER → 🏆 BEST structure wherever it fits.

## Learning Objectives

By the end of this notebook, you'll understand:
- Why you need failover and what nginx can (and cannot) do for you
- The difference between true circuit breakers and passive failover
- When to aggregate requests at the gateway vs. in a dedicated BFF
- How canary / weighted routing lets you safely ship a new version to a small % of users
- How CORS preflight requests work and how the gateway handles them
- How `X-Request-ID` enables distributed tracing across services
- Why almost every production gateway terminates TLS


## 🛠️ Setup

Make sure infrastructure is running:

```bash
cd 05-microservices/api-gateway
docker-compose up -d --build
```

### Kernel Selection

Select the `.venv` kernel in VS Code's kernel picker (top-right of notebook).
If it doesn't appear, reload the window: `Cmd+Shift+P` → "Reload Window".


In [1]:
import requests
import json
import time

GATEWAY = "http://localhost:8080"

def show(response):
    print(f"Status: {response.status_code}")
    try:
        print(json.dumps(response.json(), indent=2))
    except Exception:
        print(response.text[:300])

try:
    r = requests.get(f"{GATEWAY}/health", timeout=3)
    print(f"✅ API Gateway: {r.json()['status']}")
except Exception as e:
    print(f"❌ API Gateway not running: {e}")
    print("   Run: cd 05-microservices/api-gateway && docker-compose up -d --build")


✅ API Gateway: healthy


---

## 1️⃣ Circuit-Breaker-Style Failover

### The problem

One of your backend instances starts failing (network glitch, OOM, bad deploy). If the gateway keeps sending it traffic, every request to that instance fails. Worse: the gateway itself slows down while it waits for timeouts. This is called a **cascading failure**.

### 🚫 BAD: No failover

With a naïve load balancer, 50% of your traffic goes to the dead instance → 50% of users see errors.

### ✅ BETTER: Passive failover (what nginx gives us)

Our `nginx.conf` declares each upstream with:

```nginx
upstream user_backend {
    server user-service-1:5000 max_fails=3 fail_timeout=30s;
    server user-service-2:5000 max_fails=3 fail_timeout=30s;
}

proxy_next_upstream       error timeout http_502 http_503 http_504;
proxy_next_upstream_tries 2;
proxy_connect_timeout     2s;
proxy_read_timeout        5s;
```

Behavior:
- If a backend fails/times out, nginx retries the **next** healthy instance automatically
- After 3 failures in 30s, nginx marks the instance "down" and stops sending it traffic
- After 30s nginx tries again (very coarse "recovery" check)

### 🏆 BEST: True circuit breaker (what you'd use in production)

A real circuit breaker has three states:

```
     closed  ───failures exceed threshold──▶   open
       ▲                                         │
       │                                         │ wait timeout
       │                                         ▼
   success ◀────probe succeeds────   half-open
```

- **closed** — normal traffic flow
- **open** — all requests fail fast (no backend call made at all)
- **half-open** — let a few probe requests through; if they succeed, close; if they fail, re-open

Production tools that implement this: **Envoy**, **Istio**, **Linkerd**, **Resilience4j** (Java), **Polly** (.NET), **pybreaker** (Python). nginx OSS does not.

Let's simulate the pattern in Python so you understand the state machine:


In [2]:
# A minimal circuit breaker — the pattern your service mesh implements for you.
import time

class CircuitBreaker:
    CLOSED, OPEN, HALF_OPEN = "closed", "open", "half_open"

    def __init__(self, fail_threshold=3, recovery_seconds=5):
        self.state = self.CLOSED
        self.failures = 0
        self.opened_at = 0.0
        self.fail_threshold = fail_threshold
        self.recovery_seconds = recovery_seconds

    def call(self, fn, *args, **kwargs):
        # If OPEN, short-circuit until the recovery window passes
        if self.state == self.OPEN:
            if time.time() - self.opened_at >= self.recovery_seconds:
                self.state = self.HALF_OPEN  # let a probe request through
                print("   ↩️  HALF_OPEN: sending a probe request...")
            else:
                raise RuntimeError("circuit OPEN — failing fast")

        try:
            result = fn(*args, **kwargs)
        except Exception as exc:
            self.failures += 1
            if self.state == self.HALF_OPEN or self.failures >= self.fail_threshold:
                self.state = self.OPEN
                self.opened_at = time.time()
                print(f"   🔥 circuit OPENED after failure: {exc}")
            raise
        else:
            if self.state == self.HALF_OPEN:
                print("   ✅ probe succeeded — circuit CLOSED")
            self.state = self.CLOSED
            self.failures = 0
            return result

# Simulate: a backend that fails 5 times, then recovers
attempts = {"n": 0}
def flaky_backend():
    attempts["n"] += 1
    if attempts["n"] <= 5:
        raise RuntimeError("backend down")
    return "ok"

cb = CircuitBreaker(fail_threshold=3, recovery_seconds=2)
for i in range(16):
    try:
        out = cb.call(flaky_backend)
        print(f"Request {i+1}: state={cb.state:<10} result={out}")
    except Exception as exc:
        print(f"Request {i+1}: state={cb.state:<10} error={exc}")
    time.sleep(0.6)


Request 1: state=closed     error=backend down


Request 2: state=closed     error=backend down


   🔥 circuit OPENED after failure: backend down
Request 3: state=open       error=backend down


Request 4: state=open       error=circuit OPEN — failing fast


Request 5: state=open       error=circuit OPEN — failing fast


...

   ↩️  HALF_OPEN: sending a probe request...
   ✅ probe succeeded — circuit CLOSED
Request 12: state=closed     result=ok


Request 13: state=closed     result=ok


Request 14: state=closed     result=ok


Request 15: state=closed     result=ok


Request 16: state=closed     result=ok


### What nginx gives us vs. what a service mesh gives us

| Capability | nginx OSS (`max_fails`/`proxy_next_upstream`) | Envoy / Istio |
|-----------|:---:|:---:|
| Retry next instance on error | ✅ | ✅ |
| Eject failing instance temporarily | ✅ (coarse) | ✅ (fine-grained) |
| Open / Half-Open / Closed state machine | ❌ | ✅ |
| Fail-fast when circuit is open | ❌ | ✅ |
| Health checks (active) | ⚠️ nginx Plus only | ✅ |
| Metrics for breaker state | ❌ | ✅ |

**Takeaway:** for a small system, `max_fails` + timeouts is often enough. When you grow to many services and need production-grade resilience, move to a service mesh or a dedicated client-side breaker library.


---

## 2️⃣ Request Aggregation (Backend-For-Frontend pattern)

### The problem

A mobile "profile screen" needs: the user's details **and** their orders. Those live in two services.

### 🚫 BAD: The client makes N calls

```
Mobile app                Network
────────                  ───────
GET /api/users/1    ──────▶   (round-trip #1, ~200ms on mobile)
GET /api/orders?user_id=1 ─▶  (round-trip #2, ~200ms on mobile)
```

Two round-trips means twice the latency — painful on mobile networks — and forces the client to handle partial failures.

### 🏆 BEST: The gateway (or BFF) aggregates

```
Mobile app        Gateway / BFF          Services
────────          ─────────────          ────────
GET /api/profile/1 ─▶ fetch user ──────▶ user-service
                      fetch orders ────▶ order-service
                   ◀── combined JSON
```

One round-trip, one combined payload, and partial failures are handled server-side.

Let's hit the aggregated endpoint:


In [3]:
time.sleep(1)  # avoid rate limiting from earlier notebooks

print("🏆 Aggregated profile endpoint")
print("=" * 55)
print("ONE call to the gateway returns user + orders:")
print()

r = requests.get(f"{GATEWAY}/api/profile/1")
show(r)


🏆 Aggregated profile endpoint
ONE call to the gateway returns user + orders:

Status: 200
{
  "order_count": 2,
  "orders": [
    {
      "amount": 999.99,
      "id": "101",
      "product": "Laptop",
      "status": "shipped",
      "user_id": "1"
    },
    {
      "amount": 29.99,
      "id": "103",
      "product": "Mouse",
      "status": "processing",
      "user_id": "1"
    }
  ],
  "served_by": "user-service-2",
  "user": {
    "email": "alice@example.com",
    "id": "1",
    "name": "Alice Johnson",
    "role": "admin"
  }
}


In [4]:
# Compare: the client-side approach (what you would have WITHOUT aggregation)
# We sleep briefly between calls to stay under the gateway's 5 req/sec rate limit.

def client_side_composition(user_id):
    r1 = requests.get(f"{GATEWAY}/api/users/{user_id}")
    time.sleep(0.25)
    r2 = requests.get(f"{GATEWAY}/api/orders", params={"user_id": user_id})
    return {"user": r1.json(), "orders": r2.json().get("orders", [])}

def gateway_composition(user_id):
    return requests.get(f"{GATEWAY}/api/profile/{user_id}").json()

N = 5  # small so we don't trip the rate limiter

t0 = time.time()
for _ in range(N):
    client_side_composition("1")
    time.sleep(0.25)
client_total_ms = (time.time() - t0) / N * 1000

time.sleep(1.5)

t0 = time.time()
for _ in range(N):
    gateway_composition("1")
    time.sleep(0.25)
gateway_total_ms = (time.time() - t0) / N * 1000

# Subtract the throttle sleeps so the numbers reflect real work, not our pause.
THROTTLE_MS = 250
client_ms  = client_total_ms  - THROTTLE_MS * 2  # 2 sleeps per iteration
gateway_ms = gateway_total_ms - THROTTLE_MS * 1  # 1 sleep per iteration

print(f"Client-side (2 calls/req):  avg {client_ms:.1f} ms of network work per profile")
print(f"Gateway-side (1 call/req):  avg {gateway_ms:.1f} ms of network work per profile")
print()
print("On localhost the gap is small, but on mobile each saved round-trip")
print("is typically 100-300ms. The savings compound with more sub-calls.")


Client-side (2 calls/req):  avg 270.2 ms of network work per profile
Gateway-side (1 call/req):  avg 152.0 ms of network work per profile

On localhost the gap is small, but on mobile each saved round-trip
is typically 100-300ms. The savings compound with more sub-calls.


### Where should composition live?

| Option | Pros | Cons |
|--------|------|------|
| **In the gateway itself** (nginx + Lua / Kong) | One hop, centralized | Gateway becomes smart/stateful |
| **Dedicated BFF service** (one per client type) | Tailored per client (iOS vs Web) | Another service to own |
| **GraphQL gateway** (Apollo, Hasura) | Clients request exactly the fields they need | Schema/tooling complexity |
| **Inside a backend service** (what this lab does) | Simplest | Violates single-responsibility |

For this lab, the aggregation lives in `user_service.py` to keep the container count low. In a real system, prefer a **dedicated BFF** or a **GraphQL gateway** — especially if different client types (iOS/Android/Web) need different payloads.

### Partial-failure behavior

Notice the aggregated endpoint uses a short (1.5s) timeout when calling order-service. If that call fails, the endpoint still returns user data with `orders_unavailable: true`. This **graceful degradation** is a must-have pattern in any aggregation layer — one slow dependency should never take down the whole response.


---

## 3️⃣ Canary / Weighted Routing

### The problem

You want to ship a new version of a service but you don't want to blast 100% of real users with it on day 1. If there's a bug, everyone is impacted at once.

### 🏆 BEST: Send a small slice of traffic to the new version

A **canary release** deploys the new version alongside the old one and sends only a small percentage (e.g. 5–10%) of traffic to it. You watch dashboards: if error rate or latency stays flat, you **ramp up** the canary weight (10% → 50% → 100%). If anything looks bad, you drop it back to 0% with a single config reload — no redeploy, no panic.

```
             ┌──────────────┐
             │  API Gateway │
             └──────┬───────┘
                    │
          90% │    10% │ (canary weight)
              ▼        ▼
      ┌──────────┐  ┌──────────┐
      │  stable  │  │  canary  │   ← new version, small blast radius
      │  v1.4.0  │  │  v1.5.0  │
      └──────────┘  └──────────┘
```

### How nginx does it — weighted upstreams

Our `nginx.conf` declares:

```nginx
upstream user_canary_backend {
    server user-service-1:5000 weight=9;   # "stable" — 90% of traffic
    server user-service-2:5000 weight=1;   # "canary" — 10% of traffic
}

location /api/canary/users {
    proxy_pass http://user_canary_backend/users;
}
```

In this lab, `user-service-1` and `user-service-2` are the **same code** — we just pretend one is the new version. In production these would be two different image tags or git commits.

Let's send 100 requests through `/api/canary/users` and see the split:


In [5]:
time.sleep(1)  # avoid carry-over rate limiting

print("\U0001f424 Canary routing: ~90% stable, ~10% canary")
print("=" * 55)
print()

counts = {}
N = 100
# Small sleep between requests so we don't trip the 5r/s + burst=10 limiter.
for _ in range(N):
    r = requests.get(f"{GATEWAY}/api/canary/users")
    if r.status_code != 200:
        counts[f"err-{r.status_code}"] = counts.get(f"err-{r.status_code}", 0) + 1
        continue
    served = r.json()["served_by"]
    label = "stable (user-service-1)" if served.endswith("-1") else "canary (user-service-2)"
    counts[label] = counts.get(label, 0) + 1
    time.sleep(0.25)

print(f"Distribution across {N} requests:")
for label, count in sorted(counts.items()):
    bar = "█" * count
    pct = count / N * 100
    print(f"  {label:<30} {count:>3}  ({pct:4.1f}%)  {bar}")
print()
print("\U0001f4a1 It won't be exactly 90/10 — weighted round-robin is approximate.")
print("   With more requests the ratio gets closer to the configured weights.")


🐤 Canary routing: ~90% stable, ~10% canary



Distribution across 100 requests:
  canary (user-service-2)         10  (10.0%)  ██████████
  stable (user-service-1)         90  (90.0%)  ██████████████████████████████████████████████████████████████████████████████████████████

💡 It won't be exactly 90/10 — weighted round-robin is approximate.
   With more requests the ratio gets closer to the configured weights.


### Production canary tips

- **Start tiny** (1–5%). Enough to see bugs, small enough that impact is limited.
- **Watch the right metrics** on the canary pool specifically: error rate, p95 latency, business KPIs (checkout success rate, etc.). Don't just watch CPU.
- **Automate the rollout.** Tools like **Argo Rollouts**, **Flagger**, and **AWS CodeDeploy** auto-promote or auto-rollback based on metrics.
- **Session stickiness matters** for stateful flows. If a user's first request hits the canary, you usually want follow-ups to also hit the canary (use `ip_hash` or a cookie-based router).
- **Feature flags are the other half.** Traffic-split canaries test infra; feature flags (LaunchDarkly, Unleash) test business behavior per user. Mature teams use both.

### Close cousins of canary routing

| Pattern | How it works | When to use |
|---------|------------|-------------|
| **Blue/Green** | Deploy v2 alongside v1, flip 100% of traffic at once | Fast rollback, but full blast radius on flip |
| **Canary** | Gradually shift 1% → 10% → 50% → 100% | Safer rollout, catches issues early |
| **Shadow / Mirror** | Duplicate traffic to v2 but ignore its responses | Validate performance with zero user risk |
| **A/B Test** | Route by user segment (country, cohort, flag) | Measure business impact of a change |


---

## 4️⃣ CORS (Cross-Origin Resource Sharing)

### The problem

Your web app is served from `https://app.example.com`. Its JavaScript tries to call `https://api.example.com/users`. By default, browsers **block** this — it's a "cross-origin" request. This is a security feature called the **Same-Origin Policy**.

To allow it, the API must respond with CORS headers:

```
Access-Control-Allow-Origin: https://app.example.com
Access-Control-Allow-Methods: GET, POST
Access-Control-Allow-Headers: Content-Type, X-API-Key
```

### Preflight requests

For any request that is **not** a "simple" GET/HEAD/POST-with-simple-headers, the browser first sends an `OPTIONS` request (called a **preflight**) to ask "may I?". If the preflight response has the right CORS headers, the browser sends the real request.

```
Browser              Gateway               Backend
───────              ───────               ───────
OPTIONS /api/...  ──▶                           (never reaches backend)
                     Access-Control-Allow-*
               ◀──── 204 No Content

GET /api/...      ──▶                     ──▶   real request
               ◀──── 200 + CORS headers   ◀──
```

### 🚫 BAD: Make every backend implement CORS

Duplicated headers in every service, inconsistent rules.

### 🏆 BEST: Handle CORS at the gateway

Our nginx config does this for `/api/cors/users`:

```nginx
location /api/cors/users {
    if ($request_method = OPTIONS) {
        add_header Access-Control-Allow-Origin  "*"                              always;
        add_header Access-Control-Allow-Methods "GET, POST, PUT, DELETE, OPTIONS" always;
        add_header Access-Control-Allow-Headers "Content-Type, X-API-Key"        always;
        return 204;
    }
    add_header Access-Control-Allow-Origin "*" always;
    proxy_pass http://user_backend/users;
}
```

Let's watch both halves of the exchange:


In [6]:
print("🔎 CORS preflight (OPTIONS request)")
print("=" * 55)
r = requests.options(f"{GATEWAY}/api/cors/users",
    headers={
        "Origin": "https://app.example.com",
        "Access-Control-Request-Method": "GET",
        "Access-Control-Request-Headers": "X-API-Key",
    },
    timeout=3,
)
print(f"Status: {r.status_code}  (204 = OK, no body)")
for k, v in r.headers.items():
    if k.lower().startswith("access-control"):
        print(f"  {k}: {v}")

print()
print("🔎 Real request (GET)")
print("=" * 55)
r = requests.get(f"{GATEWAY}/api/cors/users",
                 headers={"Origin": "https://app.example.com"}, timeout=3)
print(f"Status: {r.status_code}")
for k, v in r.headers.items():
    if k.lower().startswith("access-control"):
        print(f"  {k}: {v}")


🔎 CORS preflight (OPTIONS request)
Status: 204  (204 = OK, no body)
  Access-Control-Allow-Origin: *
  Access-Control-Allow-Methods: GET, POST, PUT, DELETE, OPTIONS
  Access-Control-Allow-Headers: Content-Type, X-API-Key, X-Request-ID
  Access-Control-Max-Age: 3600

🔎 Real request (GET)
Status: 200
  Access-Control-Allow-Origin: *
  Access-Control-Allow-Methods: GET, POST, PUT, DELETE, OPTIONS
  Access-Control-Expose-Headers: X-Request-ID, X-API-Version


### Production CORS tips

- **Don't use `Access-Control-Allow-Origin: *` for credentialed requests.** If the browser sends cookies or auth headers, you must echo the exact origin back (and add `Access-Control-Allow-Credentials: true`).
- **Keep an allow-list of origins** instead of `*`. nginx `map` works well for this.
- **Set `Access-Control-Max-Age`** so browsers cache the preflight answer (reduces OPTIONS traffic).
- **Expose response headers** that JS needs to read (like `X-Request-ID`) via `Access-Control-Expose-Headers`.


---

## 5️⃣ Observability: Logging, Tracing, Monitoring

When a user says *"the app was slow yesterday"*, you need to reconstruct what happened across your services. You need **observability**:

- **Logs** — discrete events ("request X returned 500")
- **Metrics** — numbers over time ("p99 latency = 320ms")
- **Traces** — how a single request flowed through every service

The gateway is the perfect place to emit observability signals: **every** request passes through it.

### Structured access logs

Our `nginx.conf` writes JSON access logs to stdout:

```nginx
log_format gateway_json escape=json
    '{"time":"$time_iso8601",'
    '"request_id":"$request_id",'
    '"method":"$request_method",'
    '"uri":"$request_uri",'
    '"status":$status,'
    '"upstream_addr":"$upstream_addr",'
    '"upstream_response_time":"$upstream_response_time",'
    '"request_time":$request_time}';
access_log /dev/stdout gateway_json;
```

To watch the logs live while you use the notebook, open a second terminal:

```bash
docker logs -f api-gateway
```

### Distributed tracing with `X-Request-ID`

The gateway mints a unique ID for every request (nginx's built-in `$request_id`) and forwards it as `X-Request-ID`. If every service logs that ID, you can **grep across all logs** to reconstruct the journey of one request.


In [7]:
import uuid

print("📝 Sending requests so the gateway writes access logs...")
correlation_tag = str(uuid.uuid4())[:8]  # our own tag (separate from X-Request-ID)

for i in range(5):
    r = requests.get(f"{GATEWAY}/api/users",
                     headers={"User-Agent": f"traffic-bot/{correlation_tag}"})
    print(f"  req {i+1}: status={r.status_code}")

print()
print("Run this in a terminal to see the structured JSON logs for our requests:")
print()
print(f"  docker logs api-gateway 2>&1 | grep {correlation_tag}")
print()
print("You'll see one JSON line per request with: request_id, status,")
print("upstream_addr (which backend served it), and request_time.")


📝 Sending requests so the gateway writes access logs...
  req 1: status=200
  req 2: status=200
  req 3: status=200
  req 4: status=200
  req 5: status=200

Run this in a terminal to see the structured JSON logs for our requests:

  docker logs api-gateway 2>&1 | grep a443be58

You'll see one JSON line per request with: request_id, status,
upstream_addr (which backend served it), and request_time.


In [8]:
print("🧵 End-to-end trace ID propagation")
print("=" * 55)

for i in range(3):
    r = requests.get(f"{GATEWAY}/api/debug/headers")
    received = r.json()["received_headers"]
    gw_id = received.get("X-Request-Id")
    served_by = r.json()["served_by"]
    print(f"  req {i+1}: request_id={gw_id}  served_by={served_by}")

print()
print("💡 In production: log this ID in every service, then search by it in")
print("   Kibana/Datadog/Loki to see the entire trace in a single query.")
print()
print("🔗 Real tracing systems (OpenTelemetry, Jaeger, Zipkin) go further:")
print("   they correlate spans across services and show a visual timeline.")


🧵 End-to-end trace ID propagation
  req 1: request_id=994168ac3ac39f5682e0bd34c162fe66  served_by=user-service-1
  req 2: request_id=e09fee65d619bb0486f7257116992664  served_by=user-service-2
  req 3: request_id=391ec0b26f4cdbdbb6306a646c430cbf  served_by=user-service-1

💡 In production: log this ID in every service, then search by it in
   Kibana/Datadog/Loki to see the entire trace in a single query.

🔗 Real tracing systems (OpenTelemetry, Jaeger, Zipkin) go further:
   they correlate spans across services and show a visual timeline.


### Metrics to collect at the gateway

If you add nothing else to your stack, these four gateway metrics give you 90% of the value:

1. **RPS per route** — how many requests per second per endpoint
2. **Error rate per route** — percentage of 4xx/5xx responses
3. **Latency per route** — p50, p95, p99 of `$request_time`
4. **Upstream health** — how often each backend instance failed (`$upstream_status`)

Tools that ingest nginx logs/metrics easily: **Prometheus** (via `nginx-prometheus-exporter`), **Grafana**, **Datadog**, **New Relic**, **CloudWatch**.


---

## 6️⃣ SSL / TLS Termination (concept-only in this lab)

Almost every production API is served over HTTPS. The gateway is where TLS is typically **terminated**: the client talks HTTPS to the gateway, and the gateway talks plain HTTP to internal services on a private network.

```
Client ──── HTTPS (TLS 1.3) ────▶ Gateway ──── HTTP (plain) ────▶ Backend
                                   │
                                   ├── Certificate management (cert renewal, rotation)
                                   ├── TLS version & cipher enforcement
                                   └── HSTS, HTTP → HTTPS redirect
```

### 🚫 BAD: Terminate TLS in every backend

Each service needs certs, cert rotation scripts, TLS library upgrades. Cert management becomes your #1 operational nightmare.

### 🏆 BEST: Terminate once at the gateway

The gateway owns certs. Backends stay simple. Example `nginx.conf` (this is in our file as a **commented reference** — the lab doesn't ship certs so it can't run):

```nginx
server {
    listen 443 ssl http2;
    server_name api.example.com;

    ssl_certificate     /etc/ssl/certs/api.example.com.pem;
    ssl_certificate_key /etc/ssl/private/api.example.com.key;

    ssl_protocols       TLSv1.2 TLSv1.3;
    ssl_ciphers         HIGH:!aNULL:!MD5;
    ssl_prefer_server_ciphers on;

    add_header Strict-Transport-Security "max-age=31536000; includeSubDomains" always;

    location /api/users {
        proxy_pass http://user_backend/users;   # plain HTTP internally
    }
}

# Redirect any accidental plain HTTP to HTTPS
server {
    listen 80;
    server_name api.example.com;
    return 301 https://$host$request_uri;
}
```

### Where to get certs

- **Let's Encrypt** — free, automated, 90-day certs (use `certbot`)
- **AWS ACM / GCP / Azure** — free when used with their load balancers
- **Your internal CA** — for private APIs

### mTLS — when the gateway also verifies the client

Standard TLS verifies the **server**. In high-security setups (service-to-service traffic, B2B APIs), **mutual TLS (mTLS)** also verifies the **client's certificate**. nginx supports this via `ssl_client_certificate` + `ssl_verify_client on`. This is common inside service meshes (Istio enables it automatically between services).


---

## 
We used **nginx OSS** in this lab because it's free, ubiquitous, and teaches you the primitives. In production teams pick from a wider  the right choice depends on *what layer* the gateway lives at and *how much logic* you want to push into it.menu 

| Product | Type | Strengths | When to pick it |
|---------|------|-----------|-----------------|
| **nginx / nginx Plus** | Reverse proxy + gateway | Fast, simple, config-as-file | Small teams, self-hosted, want full control |
| **Envoy** | L7 proxy (C++) | True circuit breakers, retries, gRPC, observability built in | Base of service meshes; powers Istio, Consul, AWS App Mesh |
| **Kong** | API gateway (nginx + Lua plugins) | Rich plugin ecosystem (auth, transforms, quotas), admin API | Self-hosted platforms that want extensibility without writing Lua |
| **Traefik** | Cloud-native reverse proxy | Auto-discovery from Docker/Kubernetes, Let's Encrypt built in | Kubernetes-first teams, small ops footprint |
| **HAProxy** | L4/L7 load balancer | Extremely fast, rock-solid, great TCP support | Very high-RPS edges, non-HTTP protocols |
| **AWS API Gateway** | Fully managed | Pay-per-request, deep AWS integration (Lambda, IAM, Cognito, WAF) | Serverless/AWS-heavy stacks; don't want to run servers |
| **Azure API Management (APIM)** | Fully managed | Developer portal, policies-as-XML, Azure AD integration | Azure-heavy enterprises, API productization |
| **Google Cloud API Gateway / Apigee** | Managed | Apigee adds analytics, monetization, versioning UI | GCP stacks; large enterprise API programs |
| **Cloudflare / Fastly** | Edge gateway (CDN + gateway) | Global POPs, DDoS protection, WAF, Workers/compute at edge | Public APIs that need global low-latency + protection |
| **Istio / Linkerd** (Ingress Gateway) | Service mesh edge | mTLS, fine-grained traffic policies, canary + retries | Kubernetes with many services talking to each other |

### Quick heuristic

 you want *less ops* and your stack already lives there.
 you want a self-hosted gateway with a plugin ecosystem.
 you need production-grade resilience (true circuit breakers, smart retries) and you're on Kubernetes.
 you want a battle-tested L7 proxy with a small footprint and no vendor lock-in.
 you want the gateway at the *edge*, closer to users, with DDoS and WAF included.

The patterns you learned in this lab (routing, rate limiting, auth, retries, canary, CORS, TLS termination, observability) translate 1:1 to every product above. The *config syntax* differs; the *concepts* don't.



---

## 📚 Summary

### What We Learned

| Pattern | Core idea | nginx primitive | Production upgrade |
|---------|----------|-----------------|---------------------|
| Circuit-breaker-style failover | Don't send traffic to a dead instance | `max_fails`, `fail_timeout`, `proxy_next_upstream` | Envoy / Istio / Resilience4j |
| Request aggregation (BFF) | One client call = many backend calls | URL route → service that composes | Dedicated BFF / GraphQL gateway |
| Canary / weighted routing | Ship a new version to a small slice of users | `server ... weight=N;` | Argo Rollouts / Flagger / service mesh |
| CORS | Let other-origin browsers call your API | `add_header Access-Control-*` | Same, plus origin allow-list |
| Observability | Logs + traces correlated by request ID | `log_format` + `$request_id` | OpenTelemetry / Jaeger / Datadog |
| TLS termination | Encrypt at the edge, plain internally | `listen 443 ssl; ssl_certificate ...` | Managed certs (ACM / Let's Encrypt) |

### Key Takeaways

1. Basic gateways (nginx OSS) handle *most* production needs: routing, LB, rate limiting, auth, CORS, TLS, basic failover. Reach for a service mesh when you need true circuit breakers, fine-grained retries, or mTLS by default.
2. Aggregation at the gateway is a latency win on mobile. Just handle partial failures gracefully.
3. Observability is worthless without **correlation**: a `request_id` logged in every hop is the single most useful thing you can add.
4. TLS belongs at the edge. Backends shouldn't know about certificates.

### Complete Gateway Capability Matrix (all 4 notebooks)

| Notebook | Capability |
|----------|-----------|
| 1 | Path-based routing, load balancing, health checks |
| 2 | Rate limiting, API key authentication |
| 3 | Header injection, API versioning, URL rewriting |
| 4 | Circuit-breaker-style failover, request aggregation (BFF), canary routing, CORS, observability, TLS termination |

That's the complete picture of what an API gateway does in production. 🎉
